# 01 — Airbnb Prices in European Cities

**Étape du pipeline** (Ward, Grinstein et Keim) : *Raw data → Data transformation*.

Source : [Kaggle, thedevastator](https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities), 20 CSV (10 villes × semaine / week-end). On garde les **8 fichiers des 4 villes communes** avec le dataset hôtels : Londres, Berlin, Paris, Rome.

Sortie : `data/interim/airbnb.parquet`, au schéma commun (`src/prep/schema.py`, `docs/schema.md`).

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from src.prep import airbnb, schema
from src.prep.common import METRO_SEUIL_KM

## 1. Chargement des fichiers bruts

In [ ]:
brut = airbnb.charger_brut(RAW / "airbnb")
print(brut.shape)
brut.head()

In [ ]:
brut.dtypes

In [ ]:
brut.groupby(["ville_brut", "periode_brut"]).size().unstack()

**Constat** : aucune valeur manquante, et les noms de colonnes correspondent à ceux du handoff.

In [ ]:
brut.isna().sum().sum()

## 2. Distributions des variables utiles

In [ ]:
cols = ["realSum", "guest_satisfaction_overall", "cleanliness_rating", "dist", "metro_dist", "person_capacity", "bedrooms"]
brut[cols].describe(percentiles=[.01, .5, .99]).round(2)

In [ ]:
brut["room_type"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for v, g in brut.groupby("ville_brut"):
    axes[0].hist(g["realSum"] / 2, bins=80, range=(0, 600), histtype="step", label=v)
    axes[1].hist(g["metro_dist"], bins=60, range=(0, 3), histtype="step", label=v)
axes[0].set(title="Prix par nuit (realSum / 2, €)", xlabel="€")
axes[1].axvline(METRO_SEUIL_KM, color="k", ls="--")
axes[1].set(title="Distance au métro (km)", xlabel="km")
axes[0].legend(); plt.tight_layout()

**Constats**

- `realSum` est le prix total pour 2 personnes et 2 nuits. La queue de distribution est longue : le maximum dépasse 16 000 €. Les aberrants sont écartés à la fusion (notebook 03), par ville et par source.
- `guest_satisfaction_overall` est déjà sur 100.
- La part d'annonces à moins de 500 m du métro est proche de celle des hôtels qui déclarent un accès au métro (comparaison dans le notebook 03).

In [ ]:
(brut["metro_dist"] < METRO_SEUIL_KM).groupby(brut["ville_brut"]).mean().round(2)

## 3. Doublons

In [ ]:
print("lignes strictement identiques :", brut.duplicated().sum())
print("même ville, période, position et prix :", brut.duplicated(["ville_brut", "periode_brut", "lat", "lng", "realSum"]).sum())

**Constat** : aucune ligne n'est strictement identique. Quelques annonces partagent la position et le prix mais diffèrent sur d'autres colonnes, par exemple des logements différents dans un même immeuble. On les garde.

Le dataset ne contient pas d'identifiant d'annonce. On ne peut donc pas savoir si une même annonce figure à la fois en semaine et en week-end : les deux périodes restent des lignes distinctes, repérées par la colonne `periode`.

## 4. Transformation vers le schéma commun

In [ ]:
df = schema.valider(airbnb.transformer(airbnb.dedoublonner(brut)))
print(df.shape)
df.head()

In [ ]:
df.groupby(["ville", "tranche_dist"], observed=True).size().unstack()

## 5. Export intermédiaire

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM / "airbnb.parquet", index=False)
print(INTERIM / "airbnb.parquet", len(df))